# 05 - Neighborhood Scatterplot

Computes pandemic vitality (opening/closing ratio during 2020-2021) and recovery vitality (2022-2024) for each neighborhood using the neighborhood-aggregated parquet. Filters to neighborhoods with at least 500 total business events, generates an interactive scatter, and exports the resilience metrics to `pandemic_resilience.parquet`.


In [1]:
import geopandas as gpd
import plotly.graph_objects as go

gdf = gpd.read_parquet("../../data/processed/ALL_openings_closings_by_neighs_year.parquet")

covid = (
    gdf[gdf["year"].between(2020, 2021)]
    .groupby("neighborhood")[["opened", "closed"]]
    .sum().reset_index()
)
covid["covid_ratio"] = covid["opened"] / covid["closed"].replace(0, float("nan"))

recovery = (
    gdf[gdf["year"].between(2022, 2024)]
    .groupby("neighborhood")[["opened", "closed"]]
    .sum().reset_index()
)
recovery["recovery_ratio"] = recovery["opened"] / recovery["closed"].replace(0, float("nan"))

totals = (
    gdf[gdf["year"].between(2020, 2024)]
    .groupby("neighborhood")[["opened", "closed"]].sum().reset_index()
)
totals["total"] = totals["opened"] + totals["closed"]
active_neighs = totals[totals["total"] >= 500]["neighborhood"]

df = covid[["neighborhood", "covid_ratio"]].merge(
    recovery[["neighborhood", "recovery_ratio"]], on="neighborhood"
)
df = df[df["neighborhood"].isin(active_neighs)]

x_mean = df["covid_ratio"].mean()
y_mean = df["recovery_ratio"].mean()
x_min, x_max = df["covid_ratio"].min(), df["covid_ratio"].max()
y_min, y_max = df["recovery_ratio"].min(), df["recovery_ratio"].max()
x_pad = (x_max - x_min) * 0.08
y_pad = (y_max - y_min) * 0.08

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["covid_ratio"], y=df["recovery_ratio"],
    mode="markers+text", text=df["neighborhood"],
    textposition="top center",
    hovertemplate="<b>%{text}</b><br>Pandemic vitality: %{x:.2f}<br>Recovery vitality: %{y:.2f}<extra></extra>",
    marker=dict(size=12, color=df["covid_ratio"], colorscale="RdBu", showscale=True)
))

fig.add_shape(type="line", x0=x_mean, x1=x_mean, y0=y_min-y_pad, y1=y_max+y_pad,
              line=dict(dash="dash"))
fig.add_shape(type="line", x0=x_min-x_pad, x1=x_max+x_pad, y0=y_mean, y1=y_mean,
              line=dict(dash="dash"))

for x, y, text in [
    (x_max-x_pad, y_max-y_pad, "High vitality / Strong recovery"),
    (x_min+x_pad, y_max-y_pad, "Low vitality / Strong recovery"),
    (x_min+x_pad, y_min+y_pad, "Low vitality / Weak recovery"),
    (x_max-x_pad, y_min+y_pad, "High vitality / Weak recovery"),
]:
    fig.add_annotation(x=x, y=y, text=text, showarrow=False)

fig.update_layout(
    title="SF Neighborhood Business Pandemic Resilience",
    xaxis_title="Business Vitality During Pandemic (2020-2021)",
    yaxis_title="Business Vitality During Recovery (2022-2024)",
)
fig.show()

In [2]:
df_export = covid[['neighborhood', 'covid_ratio']].merge(
    recovery[['neighborhood', 'recovery_ratio']], on='neighborhood'
)
df_export = df_export[df_export['neighborhood'].isin(active_neighs)].dropna()
df_export.to_parquet('../../data/processed/pandemic_resilience.parquet', index=False)